In [ ]:
#importing the libraries
import pandas as pd
import torch
from numpy import array
import numpy as np
import os
from PIL import Image
import requests
from tqdm import tqdm
from VisionModel_utils import FrozenVisionModel_Encoding
from LangModel_utils import FrozenLanguageModel_Encoding
import json
from transform_inference import make_transform
import argparse
from datetime import datetime

In [ ]:
def pack_boolean_tensor(token_hd_vec):
    """
    Convert a boolean tensor to packed uint8 format (NumPy fallback if torch.packbits is missing).
    Optimized for CPU.
    """
    token_hd_vec_uint8 = token_hd_vec.to(torch.uint8, copy=False)

    if hasattr(torch, "packbits"): # This was available in some PyTorch versions
        try:
            return torch.packbits(token_hd_vec_uint8, dim=-1)
        except Exception:
            pass  # fallback if torch.packbits fails (rare older builds)

    arr = token_hd_vec_uint8.numpy()
    packed = np.packbits(arr, axis=-1)
    return torch.from_numpy(packed)

class HDLogitsComputer:
    """
    Fast HD logits computation with pre-allocated buffers and max pooling.

    There is a possibility of implementing this with Triton or CUDA Kernels
    to optimize memory and performance. The LUT (Look-Up Table) for popcount
    followed by sum (highlighted with arrow comment) is the main memory bottleneck.
    """
    def __init__(self, vocab_hd_packed, popcount_lut, device='cuda:0', pooling='max'):
        self.device = device
        self.vocab_hd_packed = vocab_hd_packed
        self.popcount_lut = popcount_lut
        self.vocab_size = vocab_hd_packed.shape[1]
        self.pooling = pooling  # 'max' or 'sum'
        
        # Pre-allocate buffers (reused across ALL calls)
        self.HD_logits_gpu = torch.zeros(self.vocab_size, device=device, dtype=torch.float32)
        self.HD_logits_cpu = torch.zeros(self.vocab_size, device='cuda:0', dtype=torch.float32)
        
    def compute(self, token_hd_vec, i, window_size=4, vocab_chunk_size=30000):
        start_pos = max(0, i)
        end_pos = min(self.vocab_hd_packed.shape[0], i + window_size + 1)
        window_len = end_pos - start_pos
        
        # Move token to GPU
        token_hd_vec = token_hd_vec.to(self.device, non_blocking=True)
        token_hd_expanded = token_hd_vec.unsqueeze(0).unsqueeze(0)
        
        # Reset buffer
        self.HD_logits_gpu.zero_()
        
        # Optimization: Load entire window once (window_size=4 means max 9 positions)
        if window_len <= 10:
            vocab_window = self.vocab_hd_packed[start_pos:end_pos].to(self.device, non_blocking=True)
            
            for v_start in range(0, self.vocab_size, vocab_chunk_size):
                v_end = min(v_start + vocab_chunk_size, self.vocab_size)
            
                if v_end == v_start:
                    continue  # nothing to process
            
                xor_result = torch.bitwise_xor(vocab_window[:, v_start:v_end, :], token_hd_expanded)
                hamming_distances = self.popcount_lut[xor_result.long()].sum(dim=2, dtype=torch.int32) # <--- Main memory bottleneck in the implmentation and could be improved.
            
                if self.pooling == 'max':
                    self.HD_logits_gpu[v_start:v_end] = (50000 - hamming_distances).max(dim=0)[0].float()

                else:
                    # Sum pooling: sum similarities across window
                    self.HD_logits_gpu[v_start:v_end] = (50000 - hamming_distances).sum(dim=0, dtype=torch.float32)
            
            del vocab_window
        else:
            """"
            Currently Else part is not needed as window size is small
            Also, still working on this part so might not work when used.
            Additonally, should be very slow.... 
            """
            # Fallback for larger windows
            # Fallback for larger windows
            for v_start in range(0, self.vocab_size, vocab_chunk_size):
                v_end = min(v_start + vocab_chunk_size, self.vocab_size)
                
                vocab_chunk = self.vocab_hd_packed[start_pos:end_pos, v_start:v_end, :].to(self.device, non_blocking=True)
                xor_result = torch.bitwise_xor(vocab_chunk, token_hd_expanded)
                hamming_distances = self.popcount_lut[xor_result.long()].sum(dim=2, dtype=torch.int32)
                
                # Apply pooling operation
                if self.pooling == 'max':
                    self.HD_logits_gpu[v_start:v_end] = (50000 - hamming_distances).max(dim=0)[0].float()
                else:
                    self.HD_logits_gpu[v_start:v_end] = (50000 - hamming_distances).mean(dim=0, dtype=torch.float32)
                
                del vocab_chunk, xor_result, hamming_distances
        
        # Normalize in-place (adjust divisor for max pooling)
        if self.pooling == 'max':
            self.HD_logits_gpu.div_(1.0)  # No window length normalization needed
        else:
            self.HD_logits_gpu.div_(1.0)
        
        # Copy to cuda:0 (reuse buffer)
        self.HD_logits_cpu.copy_(self.HD_logits_gpu, non_blocking=False)
        
        del token_hd_expanded
        torch.cuda.empty_cache()
        
        return self.HD_logits_gpu

def LLM_get_next_token_logits(model, tokenizer, tokens, device):
    """
    Get probabilities for the next token given a list of tokens.
    
    Args:
        model: The language model
        tokenizer: The tokenizer
        tokens: List of token IDs or a string
        device: Device to run on
    
    Returns:
        next_token_logits: Tensor of logits for all tokens in vocabulary
    """
    # Convert to token IDs if input is a string
    if isinstance(tokens, str):
        tokens = tokenizer.encode(tokens, return_tensors='pt').to(device)
    elif isinstance(tokens, list):
        tokens = torch.tensor([tokens]).to(device)
    else:
        tokens = tokens.to(device)
    
    # Get model output
    with torch.no_grad():
        outputs = model(tokens)
        logits = outputs.logits  # Shape: (batch_size, sequence_length, vocab_size)
    
    # Get logits for the last token (next token prediction)
    next_token_logits = logits[0, -1, :]  # Shape: (vocab_size,)
    
    return next_token_logits

In [ ]:
splits = {'train': 'data/train-00000-of-00001.parquet', 'validation': 'data/validation-00000-of-00001.parquet', 'test': 'data/test-00000-of-00001.parquet', 'restval': 'data/restval-00000-of-00001.parquet'}
df = pd.read_parquet("hf://datasets/yerevann/coco-karpathy/" + splits["test"])

In [ ]:
# Define device 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Initialize Frozen Vision Model
F_VM_object= FrozenVisionModel_Encoding(device = device)
    
# Initialize Frozen Language Model
F_LM_object = FrozenLanguageModel_Encoding(device = device, AutoModelForCausalLM_flag=True)

In [ ]:
# Load packed vocab predictors
#-|----------------------------------------------------|
# | Can be vocab_HD_packed_COCO or vocab_HD_packed_13M |
# | Please ensure the correct path and files are used  |
#-|----------------------------------------------------|
packed_memmap = np.memmap("/storage/group/vuh14/default/Abhishek_files/dinov3txt_qwen3/saved_HD_mats/vocab_HD_packed_COCO.dat", dtype=np.uint8, mode='r', shape=(43, 151669, 6250))
    
# Convert to torch tensor (creates view, no memory copy)
vocab_hd_packed = torch.from_numpy(packed_memmap[:]).clone()  # .clone() copies to RAM
# vocab_hd_packed = vocab_hd_packed[:21]
vocab_hd_packed = vocab_hd_packed[:41]

In [ ]:
# Create a lookup table for popcount of all byte values (0-255)
POPCOUNT_TABLE = torch.tensor([bin(i).count('1') for i in range(256)], dtype=torch.uint8)

# ============= OPTIMIZATION: Initialize once, reuse =============
vocab_hd_packed_pinned = vocab_hd_packed.pin_memory()
popcount_lut_gpu = POPCOUNT_TABLE.to(device)  # Move once, keep on GPU

# Initialize the computer once with max pooling - REUSED ACROSS ALL IMAGES
hd_computer = HDLogitsComputer(vocab_hd_packed_pinned, popcount_lut_gpu, pooling='max')

In [ ]:
vocab_hd_packed.shape

In [ ]:
transform = make_transform(resize_size=512, aspect_ratio_threshold=1.0)

In [ ]:
from semantic_clip import CLIPSemanticSampler
# Initialize sampler once (reuse across multiple captions)
sampler = CLIPSemanticSampler(
    lm_tokenizer=F_LM_object.tokenizer,
    clip_model=F_VM_object.model,
    clip_tokenizer=F_VM_object.clip_tokenizer.tokenize)

In [ ]:
def inferer_captions_using_HD(img, top_k, caption_size, window_length, fixed_temp, clip_weight):
    """
    The core function to infer captions using HD representations.
    Args:
        img: Input image (PIL Image)
        top_k: Top-k value for sampling
        caption_size: Maximum caption size
        window_length: Window length for HD computation
        fixed_temp: Fixed temperature for sampling
        clip_weight: Weight for CLIP similarity in logits computation
    Returns:
        pred_caption: Generated caption (string)
    """
    img_tensor = transform(img).unsqueeze(0)
    
    hidden_batches_imgs, clip_image_features = F_VM_object.get_h_img(img_tensor)
    clip_image_features /= clip_image_features.norm(dim=-1, keepdim=True)
    h_img = hidden_batches_imgs
    img_HD = F_VM_object.get_img_HD_vec(h_img)
    img_HD = torch.sign(img_HD)[0]
    pred_caption_tokens = [1986, 2168, 4933]
    
    i = 0
    while i < caption_size:
        
        i = len(pred_caption_tokens)-3
        if i>41:
            break
        toks, LM_rep_hidden = F_LM_object.get_h_caption(
            F_LM_object.tokenizer.decode(pred_caption_tokens)
        )

        LM_rep_HD = torch.sign(torch.matmul(LM_rep_hidden[0][-1], F_LM_object.LM_LSH_matrix))
        
        token_hd_vec = torch.sign(torch.mul(LM_rep_HD, img_HD))
        token_hd_vec = token_hd_vec > 0
        token_hd_vec = token_hd_vec.cpu()
        token_hd_vec = pack_boolean_tensor(token_hd_vec)

        HD_logits = hd_computer.compute(token_hd_vec, i, window_size=window_length)

        LLM_logits = LLM_get_next_token_logits(F_LM_object.model, F_LM_object.tokenizer, pred_caption_tokens, device)[:151669]

        HD_logits = HD_logits/HD_logits.max() + 0.15*LLM_logits/LLM_logits.max()
        
        predict_token_i = sampler.sample_next_token(
            logits=HD_logits,
            generated_tokens=pred_caption_tokens,
            clip_image_features=clip_image_features,
            temperature=fixed_temp, # CAN CHANGE PARARMS
            top_k=top_k,
            top_p=0.95,
            min_candidates= top_k,
            repetition_penalty=1.1,
            clip_weight=clip_weight
        )
                
        pred_caption_tokens.append(predict_token_i)

        clear_output(wait=True)
        print(F_LM_object.tokenizer.decode(pred_caption_tokens))
        
        if i > 1 and (pred_caption_tokens[-1] == F_LM_object.tokenizer.eos_token or pred_caption_tokens[-1] == 151645 or pred_caption_tokens[-1] == 13):
            break

        """
        One can also try the commented code below !!!
        By adding strings like " Furthermore," after encountering a fullstop
        and restarting the token prediction from i=0 for prototype matrix i.e Vocab_HD_packed
        one can obtain longer caption prediction than the prototype matrix is made for. 
        Kind of like prompting, It does work sometimes, but this wasn't evaluated/presented
        in the paper. In other words, restarting the token prediction from i=0 position 
        but having the context of the previous tokens can help generate longer but relevant captions.
        """

#        if pred_caption_tokens[-1] == 13:
#             pred_caption_tokens.append(56317)
#             pred_caption_tokens.append(11)
#             window_length=1
#             top_k = int(top_k/2)
#             i=0

#         # if pred_caption_tokens[-1] == 13:
#         #     pred_caption_tokens.append(56317)
#         #     pred_caption_tokens.append(11)
#         #     # pred_caption_tokens.append(4933)
#         #     repetition_penalty = 1.0
#         #     logits_mix_factor = logits_mix_factor / 1.5
#         #     top_k = int(top_k/2)
#         #     # i=0

#         # if i>=caption_size:
#         #     if pred_caption_tokens[-1] != 13:
#         #         pred_caption_tokens.append(13)
#         #     pred_caption_tokens.append(56317)
#         #     pred_caption_tokens.append(11)
#         #     i=0
        
#         if len(pred_caption_tokens) > 50:
#             break
#         if i > 1 and (pred_caption_tokens[-1] == F_LM_object.tokenizer.eos_token or pred_caption_tokens[-1] == 151645):
#             break
    
    pred_caption = F_LM_object.tokenizer.decode(pred_caption_tokens)
    return pred_caption

In [ ]:
F_LM_object.tokenizer.encode(" Furthermore,")

In [ ]:
# Change i for different images in df
i=70
url = df['url'][i]
image = Image.open(requests.get(url, stream=True).raw).convert("RGB")
image

In [ ]:
# Feel free to play around with the parameters for experimentation and see how it affects results
inferer_captions_using_HD(image, top_k=80, caption_size=21, window_length=2, fixed_temp=1.0, clip_weight=0.5)